[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# A Real Client &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The two cells below rebuild what the notebook set up: the module `network_client.py` first, then
the practice API, `FakeSession`, `answer` and `run_tests`. Run them first, in order. The last cell
removes the module's file again.


In [1]:
%%writefile network_client.py
"""A client for the practice API's station network.

The rest of a program imports this module, and never builds a URL, reads a status code or retries a
request. What it needs to know about the API is here: the address, how long to wait, what to send
again, what an error means, and what an answer holds.
"""

import random
import time
import uuid
from datetime import datetime

import requests
from pydantic import BaseModel, ValidationError


class Station(BaseModel):
    id: str
    name: str
    latitude: float
    longitude: float


class Reading(BaseModel):
    station: str
    time: datetime
    temperature_c: float


class ReadingsPage(BaseModel):
    readings: list[Reading]


class Plan(BaseModel):
    id: int
    name: str
    latitude: float
    longitude: float
    elevation_m: float | None = None


class APIError(Exception):
    """Anything that went wrong between this client and the API."""


class NotFoundError(APIError):
    """The API has no station or plan with that id."""


class InvalidRequestError(APIError):
    """The API refused a request as it was sent, with the problems it listed."""

    def __init__(self, message, problems=()):
        super().__init__(message)
        self.problems = list(problems)


class UnavailableError(APIError):
    """Every attempt failed: no response, a 429 or a 5xx."""


class UnexpectedResponseError(APIError):
    """The API answered with something this client was not written to read."""


class NetworkClient:
    """The station network's API as methods, which return checked models or raise an APIError."""

    RETRY_STATUSES = {429, 500, 502, 503, 504}

    def __init__(self, base_url, session=None, attempts=3, timeout=(3.05, 10), sleep=time.sleep,
                 seed=None):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session() if session is None else session
        self.attempts = attempts
        self.timeout = timeout
        self.sleep = sleep
        self.rng = random.Random(seed)

    def station(self, station_id):
        """One station."""
        return self._parse(Station, self._send("GET", f"/stations/{station_id}"))

    def readings(self, station, per_page=100):
        """Every reading from a station. Each page is fetched when the loop reaches it."""
        response = self._send("GET", "/network/readings", params={"station": station, "per_page": per_page})
        while True:
            for reading in self._parse(ReadingsPage, response).readings:
                yield reading
            if "next" not in response.links:
                return
            response = self._send("GET", response.links["next"]["url"])

    def create_plan(self, name, latitude, longitude, **fields):
        """A new plan, created once however many attempts it takes."""
        plan = {"name": name, "latitude": latitude, "longitude": longitude, **fields}
        return self._parse(Plan, self._send("POST", "/network/plans", json=plan))

    def get(self, path, **params):
        """The JSON at a path this client has no method for, with the same retries and errors."""
        response = self._send("GET", path, params=params)
        try:
            return response.json()
        except requests.JSONDecodeError as error:
            raise UnexpectedResponseError(f"GET {path} did not answer with JSON") from error

    def _send(self, method, path, **kwargs):
        """The response to a request, from the first attempt that gets one worth keeping."""
        url = path if path.startswith("http") else self.base_url + path    # a Link header's address is whole
        headers = {"X-Request-Id": uuid.uuid4().hex}                       # one id for every attempt
        if method == "POST":
            headers["Idempotency-Key"] = str(uuid.uuid4())                 # so a retry cannot create twice
        for attempt in range(1, self.attempts + 1):
            response = None
            try:
                response = self.session.request(method, url, headers=headers, timeout=self.timeout, **kwargs)
            except (requests.ConnectionError, requests.Timeout) as error:
                outcome = type(error).__name__
            else:
                if response.status_code not in self.RETRY_STATUSES:
                    self._raise_for(response)
                    return response
                outcome = response.status_code
            if attempt == self.attempts:
                raise UnavailableError(f"{method} {path}: {outcome} on attempt {attempt} of {self.attempts}")
            self.sleep(self._wait(attempt, response))

    def _wait(self, attempt, response):
        """Seconds before the next attempt: what Retry-After asks for, or a backoff with jitter."""
        retry_after = "" if response is None else response.headers.get("Retry-After", "")
        if retry_after.isdigit():
            return float(retry_after)
        return self.rng.uniform(0, 2 ** (attempt - 1))

    def _raise_for(self, response):
        """Raise the APIError that says what went wrong, if anything did."""
        if response.status_code < 400:
            return
        body = response.json() if "json" in response.headers.get("Content-Type", "") else {}
        message = body.get("error", f"{response.status_code} {response.reason}")
        if response.status_code == 404 and not message.startswith("nothing at"):
            raise NotFoundError(message)
        if response.status_code in (400, 422):
            raise InvalidRequestError(message, body.get("problems", []))
        raise APIError(f"{response.status_code}: {message}")

    def _parse(self, model, response):
        """The body as the model, checked strictly: an UnexpectedResponseError if it does not fit."""
        try:
            return model.model_validate_json(response.content, strict=True)
        except ValidationError as error:
            found = [f"{'.'.join(map(str, problem['loc'])) or 'the body'}: {problem['msg']}"
                     for problem in error.errors()]
            raise UnexpectedResponseError(f"{model.__name__}: {'; '.join(found)}") from error


Writing network_client.py


In [2]:
import importlib
import json
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier
import network_client
importlib.reload(network_client)
from network_client import InvalidRequestError, NetworkClient, NotFoundError


class FakeSession:
    """Stands in for requests.Session: answers with the answers it was given, in order, and keeps every request."""

    def __init__(self, *answers):
        self.answers = list(answers)
        self.requests = []

    def request(self, method, url, **kwargs):
        self.requests.append((method, url, kwargs))
        answer = self.answers.pop(0)
        if isinstance(answer, Exception):
            raise answer
        return answer


def answer(status, body, headers=None):
    """A response with a status code, a JSON body and any other headers, built with no server."""
    response = requests.Response()
    response.status_code = status
    response.headers.update({"Content-Type": "application/json", **(headers or {})})
    response._content = json.dumps(body).encode()          # where a Response keeps its body
    return response


def run_tests(*tests):
    """Run tests, and report every one that fails, not only the first."""
    failed = 0
    for test in tests:
        try:
            test()
        except Exception as error:
            failed += 1
            print(f"FAILED {test.__name__}: {type(error).__name__}: {error}")
        else:
            print(f"passed {test.__name__}")
    print(f"{len(tests) - failed} passed, {failed} failed")


TROMSO = {"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}

BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A station, as an object.


In [3]:
client = NetworkClient(BASE)
oslo = client.station("oslo")

print(oslo.name, oslo.latitude, oslo.longitude)


Oslo 59.91 10.75


`station` returns a `Station`, so its fields are attributes, and the latitude and the longitude are
already numbers.


**2.** The exception for a station that is not there.


In [4]:
try:
    client.station("bodo")
except NotFoundError as error:
    print(type(error).__name__, "|", error)


NotFoundError | no station with id 'bodo'


The body said no station has that id, so the client raised `NotFoundError`, which a program that
skips missing stations can catch without catching any other failure.


**3.** Every reading, 25 to a page.


In [5]:
temperatures = [reading.temperature_c for reading in client.readings("bergen", per_page=25)]

print(len(temperatures), "readings | lowest:", min(temperatures))


72 readings | lowest: 1.2


72 readings, from three pages of 25, 25 and 22. The list comprehension never saw a page: `readings`
followed the `next` addresses while the comprehension asked for more.


**4.** A plan, and a plan with a problem.


In [6]:
print(client.create_plan("Alta", 69.97, 23.27))

try:
    client.create_plan("Kautokeino", 95, 23.04)
except InvalidRequestError as error:
    for problem in error.problems:
        print(f"  {problem['field']}: {problem['problem']}")


id=1 name='Alta' latitude=69.97 longitude=23.27 elevation_m=None
  latitude: must be a number from -90 to 90


The first plan came back as a `Plan`, with `elevation_m` as `None`, the default its model gives a
field the API did not send. The second got `422`, and `InvalidRequestError` carried the API's list
of problems.


**5.** A fake 503, and the wait it asked for.


In [7]:
waits = []
fake = FakeSession(answer(503, {"error": "the service is busy: try again"}, {"Retry-After": "2"}), answer(200, TROMSO))
client_with_fake = NetworkClient("http://api.test", session=fake, sleep=waits.append)

print(client_with_fake.station("tromso").name, waits)


Tromso [2.0]


One wait, of the 2 seconds `Retry-After` asked for, noted and not waited, so the cell finished at
once.


**6.** A test for the problems in a 422.


In [8]:
def test_a_422_lists_its_problems():
    body = {"error": "the plan has problems",
            "problems": [{"field": "latitude", "problem": "must be a number from -90 to 90"}]}
    fake = FakeSession(answer(422, body))
    try:
        NetworkClient("http://api.test", session=fake).create_plan("Kautokeino", 95, 23.04)
    except InvalidRequestError as error:
        assert [problem["field"] for problem in error.problems] == ["latitude"]
    else:
        raise AssertionError("a 422 raised nothing")


run_tests(test_a_422_lists_its_problems)


passed test_a_422_lists_its_problems
1 passed, 0 failed


The fake's `422` never reached a server, and the test checks the one behavior its name promises. To
see that it can fail, give the fake a `201` with a plan instead, and it reports `FAILED`.

Last, remove the module's file, as the notebook does:


In [9]:
Path("network_client.py").unlink()

print("network_client.py still there:", Path("network_client.py").exists())


network_client.py still there: False


---

&#8592; **Back to:** [A Real Client](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/14-a-real-client.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
